# K-Means and Hierarchical Clustering on Sales Data

This notebook:
1. Loads `sales_data_sample.csv`
2. Preprocesses the customer data
3. Uses the **Elbow Method** to determine a suitable number of clusters
4. Performs **K-Means clustering**
5. Performs **Hierarchical (Agglomerative) clustering**
6. Visualizes and compares the resulting clusters

In [ ]:
!pip -q install pandas numpy matplotlib seaborn scikit-learn scipy

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import files

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score

from scipy.cluster.hierarchy import dendrogram, linkage

import warnings
warnings.filterwarnings("ignore")


In [ ]:
# Upload sales_data_sample.csv
uploaded = files.upload()


In [ ]:
# Load dataset
file_name = next(iter(uploaded))
df = pd.read_csv(file_name)

print("Dataset shape:", df.shape)
display(df.head())


## 1. Data Preprocessing

In [ ]:
print("Dataset information:")
df.info()

print("\nMissing values:")
print(df.isnull().sum())


In [ ]:
# Display descriptive statistics
display(df.describe(include="all"))


In [ ]:
# Remove duplicate rows
print("Duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates()

print("Shape after removing duplicates:", df.shape)


## 2. Select Clustering Features

For customer segmentation, the important numerical variables are:

- Customer Age
- Annual Income
- Spending Score

`Customer ID` is only an identifier, so it should not be used for clustering. `Customer Gender` is categorical and is not required for this clustering analysis.

In [ ]:
# Find the actual column names in the uploaded dataset
print(df.columns.tolist())


In [ ]:
# Adjust these names if your CSV uses different column names.
feature_columns = [
    "Customer Age",
    "Annual Income",
    "Spending Score"
]

# Handle common alternative column names
column_map = {
    "Age": "Customer Age",
    "Annual Income (k$)": "Annual Income",
    "Annual Income (in Thousand Dollars)": "Annual Income",
    "Spending Score (1-100)": "Spending Score"
}

for old_name, new_name in column_map.items():
    if old_name in df.columns and new_name not in df.columns:
        df = df.rename(columns={old_name: new_name})

feature_columns = [
    "Customer Age",
    "Annual Income",
    "Spending Score"
]

X = df[feature_columns].copy()

# Convert to numeric and remove invalid rows
for column in feature_columns:
    X[column] = pd.to_numeric(X[column], errors="coerce")

X = X.dropna()

print("Selected features:")
display(X.head())
print("Feature shape:", X.shape)


## 3. Scale the Features

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Feature scaling completed.")


## 4. Determine Number of Clusters using Elbow Method

In [ ]:
inertia = []
k_values = range(1, 11)

for k in k_values:
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    kmeans.fit(X_scaled)
    inertia.append(kmeans.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(k_values, inertia, marker="o")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Within-Cluster Sum of Squares (Inertia)")
plt.title("Elbow Method")
plt.xticks(list(k_values))
plt.grid(True)
plt.show()


### Select K

Look for the point where the decrease in inertia starts becoming much smaller. This point is the **elbow** and provides a practical choice for the number of clusters.

For this customer-segmentation dataset, `K=5` is commonly used, but the elbow plot above should be used to verify the choice for the uploaded dataset.

In [ ]:
# Set the number of clusters based on the elbow plot
k = 5

print("Selected number of clusters:", k)


## 5. K-Means Clustering

In [ ]:
kmeans = KMeans(
    n_clusters=k,
    random_state=42,
    n_init=10
)

kmeans_labels = kmeans.fit_predict(X_scaled)

kmeans_result = X.copy()
kmeans_result["KMeans_Cluster"] = kmeans_labels

display(kmeans_result.head())


In [ ]:
# K-Means cluster sizes
print("K-Means cluster sizes:")
print(kmeans_result["KMeans_Cluster"].value_counts().sort_index())


In [ ]:
# Cluster centers in original units
centers_scaled = kmeans.cluster_centers_
centers_original = scaler.inverse_transform(centers_scaled)

centers_df = pd.DataFrame(
    centers_original,
    columns=feature_columns
)

centers_df.index.name = "Cluster"

print("K-Means Cluster Centers:")
display(centers_df)


## 6. Visualize K-Means Clusters

In [ ]:
plt.figure(figsize=(8, 6))

sns.scatterplot(
    data=kmeans_result,
    x="Annual Income",
    y="Spending Score",
    hue="KMeans_Cluster",
    palette="tab10",
    s=80
)

plt.title("K-Means Customer Segmentation")
plt.xlabel("Annual Income")
plt.ylabel("Spending Score")
plt.legend(title="Cluster")
plt.show()


## 7. K-Means Silhouette Score

In [ ]:
kmeans_silhouette = silhouette_score(X_scaled, kmeans_labels)

print("K-Means Silhouette Score:", round(kmeans_silhouette, 4))


## 8. Hierarchical Clustering Dendrogram

In [ ]:
# Ward linkage for hierarchical clustering
linkage_matrix = linkage(X_scaled, method="ward")

plt.figure(figsize=(12, 6))
dendrogram(
    linkage_matrix,
    truncate_mode="lastp",
    p=30,
    leaf_rotation=90,
    leaf_font_size=10
)

plt.title("Hierarchical Clustering Dendrogram")
plt.xlabel("Customers / Cluster Groups")
plt.ylabel("Distance")
plt.show()


## 9. Agglomerative Hierarchical Clustering

In [ ]:
hierarchical = AgglomerativeClustering(
    n_clusters=k,
    linkage="ward"
)

hierarchical_labels = hierarchical.fit_predict(X_scaled)

hierarchical_result = X.copy()
hierarchical_result["Hierarchical_Cluster"] = hierarchical_labels

display(hierarchical_result.head())


In [ ]:
print("Hierarchical cluster sizes:")
print(
    hierarchical_result["Hierarchical_Cluster"]
    .value_counts()
    .sort_index()
)


## 10. Visualize Hierarchical Clusters

In [ ]:
plt.figure(figsize=(8, 6))

sns.scatterplot(
    data=hierarchical_result,
    x="Annual Income",
    y="Spending Score",
    hue="Hierarchical_Cluster",
    palette="tab10",
    s=80
)

plt.title("Hierarchical Customer Segmentation")
plt.xlabel("Annual Income")
plt.ylabel("Spending Score")
plt.legend(title="Cluster")
plt.show()


## 11. Hierarchical Clustering Silhouette Score

In [ ]:
hierarchical_silhouette = silhouette_score(
    X_scaled,
    hierarchical_labels
)

print(
    "Hierarchical Clustering Silhouette Score:",
    round(hierarchical_silhouette, 4)
)


## 12. Compare Both Methods

In [ ]:
comparison = pd.DataFrame({
    "Method": ["K-Means", "Hierarchical"],
    "Number of Clusters": [k, k],
    "Silhouette Score": [
        kmeans_silhouette,
        hierarchical_silhouette
    ]
})

display(comparison)


## Conclusion

K-Means and Agglomerative Hierarchical Clustering were implemented on the customer sales dataset. The Elbow Method was used to identify a suitable value of `K`, and the resulting customer groups were visualized using Annual Income and Spending Score. Silhouette scores were also calculated to measure cluster separation.